## 1. Data Processing

### Step 1: 

In [3]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURATION ---
GAP_THRESHOLD = 3600       # 1 hour: Split data if gap is larger than this
TARGET_RATE = 10.0         # 10 Hz: Sufficient for "water hammer" effects [cite: 33]
DT = 1.0 / TARGET_RATE     # 0.1s time step

# Window Definition for Task 2 (Anomaly Detection)
# We need the transition (0/0 state) AND the "tail toward steady state" 
BUFFER_PRE = 10.0          # 10s before valve starts moving
BUFFER_POST = 60.0         # 60s after valve closes to capture pressure decay

# Autoencoder Input Size
# ~120s max transition + 10s pre + 60s post = 190s * 10Hz = 1900 points.
FIXED_LENGTH_POINTS = 2000 

# Signals to use as INPUT (X) for the Autoencoder
# Note: Valve signals are EXCLUDED from model input as per instructions [cite: 69]
INPUT_SIGNALS = [
    "active_power",
    "guide_vane_position",
    "water_pressure_downstream",
    "water_pressure_upstream"
]

# Signals used ONLY for defining start/end times
CONTROL_SIGNALS = ["ball_valve_closed", "ball_valve_open"]

# Paths
DATA_DIR = 'data' # Update this to your folder path

### Step 2: Load data and fix Sarelli sign

In [7]:
def load_and_prep_raw(file_path, mapping_df):
    """
    Loads parquet, merges mapping, and fixes Sarelli sign convention.
    """
    df = pd.read_parquet(file_path)
    
    # Merge mapping
    df = df.merge(mapping_df, on="signal_id", how="left")
    
    # Convert ns timestamp to seconds (float)
    df['ts_sec'] = df['ts'].astype(np.int64) // 10**9
    df['ts_sec'] = df['ts_sec'] + (df['ts'].dt.microsecond / 1e6)
    
    # Fix Sarelli: "active_power is always negative" -> Flip to positive 
    # We do this to align it with Mapragg's turbine mode for the model.
    if "Sarelli" in file_path or (len(df) > 0 and df['stage'].iloc[0] == 'Sarelli'):
        mask = df['signal_name'] == 'active_power'
        df.loc[mask, 'value'] *= -1
        
    return df.sort_values('ts_sec')

# Load Mapping
df_mapping = pd.read_csv('signal_descriptions.csv') 
print("Setup functions ready.")

Setup functions ready.


### Step 3: The Resampler (with Gap Handling)

In [5]:
def resample_chunk(df_chunk):
    """
    Resamples a continuous dataframe to a fixed uniform grid (10Hz).
    - Analog signals: Linear Interpolation 
    - Binary signals: Zero-Order Hold (Step function)
    """
    if df_chunk.empty: return None

    # Create uniform time grid
    t_start = df_chunk['ts_sec'].min()
    t_end = df_chunk['ts_sec'].max()
    t_new = np.arange(t_start, t_end, DT)
    
    resampled_data = {'ts_sec': t_new}
    
    for sig in INPUT_SIGNALS + CONTROL_SIGNALS:
        sub = df_chunk[df_chunk['signal_name'] == sig].sort_values('ts_sec')
        if sub.empty: continue
            
        t_raw = sub['ts_sec'].values
        v_raw = sub['value'].values
        
        # Interpolation Logic [cite: 111]
        if sig in CONTROL_SIGNALS:
            # Binary: Forward fill (Step)
            idx = np.searchsorted(t_raw, t_new, side='right') - 1
            idx = np.clip(idx, 0, len(v_raw)-1)
            resampled_data[sig] = v_raw[idx]
        else:
            # Analog: Linear Interpolation
            resampled_data[sig] = np.interp(t_new, t_raw, v_raw)
            
    return pd.DataFrame(resampled_data)

### Step 4: Visualization

In [6]:
def verify_resampling(raw_df, resampled_df, signal_name, segment_idx=0, duration=300):
    """
    Plots Raw vs Resampled to check accuracy.
    """
    # Filter Raw
    raw_sig = raw_df[raw_df['signal_name'] == signal_name]
    
    # Get a slice of time from the resampled data
    start_time = resampled_df['ts_sec'].min() + (segment_idx * duration)
    end_time = start_time + duration
    
    res_slice = resampled_df[(resampled_df['ts_sec'] >= start_time) & 
                             (resampled_df['ts_sec'] <= end_time)]
    
    raw_slice = raw_sig[(raw_sig['ts_sec'] >= start_time) & 
                        (raw_sig['ts_sec'] <= end_time)]
    
    plt.figure(figsize=(15, 5))
    
    # Plot Resampled (Red Line)
    plt.plot(res_slice['ts_sec'], res_slice[signal_name], 
             color='red', linewidth=2, alpha=0.7, label='Resampled (10Hz)')
    
    # Plot Raw (Blue Dots)
    plt.scatter(raw_slice['ts_sec'], raw_slice['value'], 
                color='blue', s=20, alpha=0.6, label='Raw Data')
    
    plt.title(f"Data Integrity Check: {signal_name}")
    plt.xlabel("Time (sec)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("Visualization tool ready. Call verify_resampling() inside the loop to check.")

Visualization tool ready. Call verify_resampling() inside the loop to check.


### Step 5: Event Extraction Logic

In [9]:
def extract_closing_events(df_resampled):
    """
    Scans a resampled dataframe for Closing Events.
    Closing defined as: Valve was OPEN, then enters TRANSITION (0/0), then CLOSED.
    """
    events = []
    
    closed = df_resampled['ball_valve_closed'].round().astype(int)
    opened = df_resampled['ball_valve_open'].round().astype(int)
    
    # Transition State (Both 0)
    in_trans = ((closed == 0) & (opened == 0)).astype(int)
    
    # Find blocks of transitions
    changes = np.diff(np.concatenate(([0], in_trans.values, [0])))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]
    
    for s, e in zip(starts, ends):
        # Filter noise
        if (e - s) < (0.5 * TARGET_RATE): continue
        
        # Check if this is a CLOSING event
        # Look 1 step before the transition started
        idx_prev = max(0, s - 1)
        prev_open = opened.iloc[idx_prev]
        
        # If it wasn't open before, it's not a closing event (might be opening)
        if prev_open != 1: continue 
            
        # Define window with buffers (Tail for pressure decay)
        idx_start = max(0, s - int(BUFFER_PRE * TARGET_RATE))
        idx_end   = min(len(df_resampled), e + int(BUFFER_POST * TARGET_RATE))
        
        window = df_resampled.iloc[idx_start : idx_end].copy()
        
        # Check completeness
        if not set(INPUT_SIGNALS).issubset(window.columns): continue

        events.append({
            "data": window[INPUT_SIGNALS].to_numpy(),
            "duration": (e - s) * DT, # Task 1 Metric
            "ts_start": window['ts_sec'].iloc[0]
        })
        
    return events

### Step 6: Main Processing Loop 
#### Phase 1: Split and store

In [4]:
import shutil
import os


# --- CONFIGURATION ---
GAP_THRESHOLD = 3600  # 1 hour [cite: 108]
DATA_DIR = 'data'     # Your raw data folder
OUTPUT_DIR = 'data/interim_segments' # Where to store the split pieces

# Reset output directory to ensure no old data remains
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# [cite_start]Get list of raw files [cite: 101]
parquet_files = sorted([os.path.join(DATA_DIR, f) 
                        for f in os.listdir(DATA_DIR) 
                        if f.endswith('.parquet') and 'real_measurements' in f])

print(f"Found {len(parquet_files)} raw files to process.")

segment_registry = []

for f_path in parquet_files:
    file_name = os.path.basename(f_path)
    print(f"\nProcessing: {file_name}")
    
    # [cite_start]1. Load Raw Data [cite: 102]
    try:
        df = pd.read_parquet(f_path)
    except Exception as e:
        print(f"Error reading {file_name}: {e}")
        continue

    # 2. Merge Mapping (Optional, but good to have names attached)
    # Assuming df_mapping is available from your previous cells
    if 'df_mapping' in globals():
        df = df.merge(df_mapping, on="signal_id", how="left")
    
    # 3. Fix Sarelli Power Sign (User Instruction)
    # We do this NOW so all saved segments are physically correct
    if "Sarelli" in file_name or (len(df) > 0 and 'stage' in df.columns and df['stage'].iloc[0] == 'Sarelli'):
        if 'signal_name' in df.columns:
            mask = df['signal_name'] == 'active_power'
            df.loc[mask, 'value'] *= -1
            print("  -> Fixed Sarelli power sign.")

    # 4. Convert Timestamp to Seconds (Float) for Gap Calculation
    # [cite: 107] 'ts' is timestamp at time of measuring
    df['ts_sec'] = df['ts'].astype(np.int64) // 10**9
    df['ts_sec'] = df['ts_sec'] + (df['ts'].dt.microsecond / 1e6)
    df = df.sort_values('ts_sec')

    # 5. Identify Gaps
    # Calculate difference between consecutive rows
    # Note: Since data is long-format (mixed signals), a 'gap' is only a true gap 
    # if NO signal was recorded for > 1 hour.
    time_diffs = df['ts_sec'].diff()
    
    # Create a Group ID that increments every time a large gap is found
    df['segment_group'] = (time_diffs > GAP_THRESHOLD).cumsum()
    
    # 6. Split and Save
    grouped = df.groupby('segment_group')
    
    count = 0
    for grp_id, segment_df in grouped:
        # Filter: Ignore tiny segments (e.g., less than 5 minutes of data or < 100 points)
        duration = segment_df['ts_sec'].max() - segment_df['ts_sec'].min()
        if len(segment_df) < 100 or duration < 300:
            continue
            
        # Create a unique filename for this segment
        seg_filename = f"{file_name.replace('.parquet', '')}_seg{grp_id:03d}.pkl"
        save_path = os.path.join(OUTPUT_DIR, seg_filename)
        
        # Save to disk
        segment_df.to_pickle(save_path)
        
        # Log metadata
        segment_registry.append({
            "original_file": file_name,
            "segment_file": seg_filename,
            "start_ts": segment_df['ts_sec'].min(),
            "end_ts": segment_df['ts_sec'].max(),
            "duration_hours": duration / 3600.0,
            "num_points": len(segment_df)
        })
        count += 1
        
    print(f"  -> Split into {count} valid segments.")

# Save the registry so we know what we have
pd.DataFrame(segment_registry).to_csv(os.path.join(OUTPUT_DIR, 'segment_registry.csv'), index=False)
print(f"\nDone! All segments stored in '{OUTPUT_DIR}'.")

Found 10 raw files to process.

Processing: Mapragg_MG1_testing_real_measurements.parquet
  -> Split into 3 valid segments.

Processing: Mapragg_MG1_training_real_measurements.parquet
  -> Split into 16 valid segments.

Processing: Mapragg_MG2_testing_real_measurements.parquet
  -> Split into 3 valid segments.

Processing: Mapragg_MG2_training_real_measurements.parquet
  -> Split into 16 valid segments.

Processing: Mapragg_MG3_testing_real_measurements.parquet
  -> Split into 3 valid segments.

Processing: Mapragg_MG3_training_real_measurements.parquet
  -> Split into 16 valid segments.

Processing: Sarelli_MG1_testing_real_measurements.parquet
  -> Split into 3 valid segments.

Processing: Sarelli_MG1_training_real_measurements.parquet
  -> Split into 16 valid segments.

Processing: Sarelli_MG2_testing_real_measurements.parquet
  -> Split into 3 valid segments.

Processing: Sarelli_MG2_training_real_measurements.parquet
  -> Split into 16 valid segments.

Done! All segments stored in

#### Phase 2: Load, resample, Extract

In [ ]:
# Ensure these configuration variables are defined from previous cells
# DT = 0.1 (or 1.0/TARGET_RATE)
# INPUT_SIGNALS = ["active_power", "guide_vane_position", "water_pressure_downstream", "water_pressure_upstream"]
# CONTROL_SIGNALS = ["ball_valve_closed", "ball_valve_open"]

def resample_segment(df_seg, rate=10.0):
    """
    Resamples a Loaded Segment to a fixed Hz.
    Analog -> Linear Interpolation, Binary -> Step Function.
    
    This helps transform raw data into a format suitable for training[cite: 110].
    """
    if df_seg.empty: return None
    
    # Create uniform time grid
    t_start = df_seg['ts_sec'].min()
    t_end = df_seg['ts_sec'].max()
    t_new = np.arange(t_start, t_end, 1.0/rate)
    
    resampled = {'ts_sec': t_new}
    
    for sig in INPUT_SIGNALS + CONTROL_SIGNALS:
        # Get raw signal data
        sub = df_seg[df_seg['signal_name'] == sig].sort_values('ts_sec')
        if sub.empty: continue
        
        t_raw = sub['ts_sec'].values
        v_raw = sub['value'].values
        
        # Interpolation Logic [cite: 111]
        if sig in CONTROL_SIGNALS:
            # Binary: Forward fill / Zero-Order Hold to avoid data leakage
            idx = np.searchsorted(t_raw, t_new, side='right') - 1
            idx = np.clip(idx, 0, len(v_raw)-1)
            resampled[sig] = v_raw[idx]
        else:
            # Analog: Linear Interpolation
            resampled[sig] = np.interp(t_new, t_raw, v_raw)
            
    return pd.DataFrame(resampled)

# --- PHASE 2: PROCESSING & VISUALIZATION (CORRECTED) ---

# Ensure df_mapping is defined (from your initial setup)
if 'df_mapping' not in globals():
    raise ValueError("df_mapping is missing! Please run the cell defining the mapping dictionary first.")

SEGMENT_DIR = 'data/interim_segments'
processed_events = []
segment_files = sorted([f for f in os.listdir(SEGMENT_DIR) if f.endswith('.pkl')])

print(f"Processing {len(segment_files)} segments...")

for i, seg_file in enumerate(segment_files):
    # 1. Load One Segment
    load_path = os.path.join(SEGMENT_DIR, seg_file)
    df_raw_seg = pd.read_pickle(load_path)
    
    # === FIX START: Restore Missing Columns ===
    # If signal_name is missing, merge it back now
    if 'signal_name' not in df_raw_seg.columns:
        df_raw_seg = df_raw_seg.merge(df_mapping, on='signal_id', how='left')
        
    # Check Sarelli Sign: If Stage is Sarelli and Power is still Negative (Raw), flip it
    # [cite: 87] active_power for Sarelli is always negative in raw data
    if len(df_raw_seg) > 0 and 'stage' in df_raw_seg.columns and df_raw_seg['stage'].iloc[0] == 'Sarelli':
        # Check mean of active_power to see if it's still negative
        ap_mask = df_raw_seg['signal_name'] == 'active_power'
        if not df_raw_seg[ap_mask].empty:
            mean_val = df_raw_seg.loc[ap_mask, 'value'].mean()
            if mean_val < 0: # It implies it hasn't been fixed yet
                df_raw_seg.loc[ap_mask, 'value'] *= -1
    # === FIX END ===

    # 2. Resample (Now this will work because signal_name exists)
    # [cite: 110] Transform raw data for training
    df_clean = resample_segment(df_raw_seg, rate=10.0)
    if df_clean is None: continue
        
    # 3. Extract Transitions
    events = extract_closing_events(df_clean)
    
    for ev in events:
        # --- FIX: Add the filename to the dictionary ---
        ev['source_segment'] = seg_file
        processed_events.append(ev)

print(f"Done. Extracted {len(processed_events)} total closing events.")

Processing 95 segments...
Done. Extracted 2985 total closing events.


#### Visualization

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# Define the feature names corresponding to the columns in your 'data' array
# These must match the INPUT_SIGNALS list you used during extraction
FEATURE_NAMES = [
    "active_power", 
    "guide_vane_position", 
    "water_pressure_downstream", 
    "water_pressure_upstream"
]

def plot_closing_event(event_index):
    if not processed_events:
        print("No events found in processed_events list.")
        return
        
    # Get the event
    ev = processed_events[event_index]
    data = ev['data']
    
    # --- FIX: Use .get() to avoid KeyError ---
    source_name = ev.get('source_segment', 'Unknown Source')
    
    # Create Time Axis (0 to N seconds)
    # We use DT (0.1s) from your configuration
    time_axis = np.arange(len(data)) * 0.1 
    
    # Setup Plot
    fig, axes = plt.subplots(len(FEATURE_NAMES), 1, figsize=(10, 8), sharex=True)
    
    # Metadata Title
    fig.suptitle(f"Event #{event_index} | Duration: {ev['duration']:.2f}s | Source: {source_name}", fontsize=14)
    
    for i, feature in enumerate(FEATURE_NAMES):
        ax = axes[i]
        signal_values = data[:, i]
        
        # Plot signal
        ax.plot(time_axis, signal_values, linewidth=1.5)
        ax.set_ylabel(feature, fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Visual Checks for "Closing"
        if feature == "active_power":
            ax.set_title("Check: Does Power drop?", loc='left', fontsize=8, color='gray')
        if "pressure" in feature:
            ax.set_title("Check: Is there a Water Hammer (spike)?", loc='left', fontsize=8, color='gray')

    axes[-1].set_xlabel("Time (seconds) from Window Start")
    plt.tight_layout()
    plt.show()

# Create Interactive Slider
interact(plot_closing_event, 
         event_index=widgets.IntSlider(min=0, max=len(processed_events)-1, step=1, value=0));

interactive(children=(IntSlider(value=0, description='event_index', max=2984), Output()), _dom_classes=('widge…